# Kaggle Scientific Smoke V2 — Dependency-Aware Selective Regeneration Benchmark

**SCIENTIFIC SMOKE V2 / NON-PUBLICATION**

Runs the minimal real Kaggle smoke using the KaggleQwenBackend on GPU.

- **Scientific Smoke V2**: 1 repository (todo) × 3 frozen scenarios (todo-smoke-001/002/003) × 3 arms × 1 run = 9 total runs
- **Arms**: monolithic, selective, iterative_repository_agent
- **Backend**: kaggle-qwen (Qwen2.5-Coder on Kaggle GPU)
- **OpenRouter**: NOT used for this smoke

Use only --profile scientific-smoke-v2. Do not switch to Pilot or Research.


In [ ]:
import os
import sys
import subprocess
import zipfile
import json as _json
from datetime import datetime
from pathlib import Path

# ---- Discover Kaggle Datasets ----------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")

KNOWN_CODE = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-code"
KNOWN_DATA = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-data"
KNOWN_MODEL = KAGGLE_INPUT / "models/qwen-lm/qwen2.5-coder/transformers/14b-instruct/1"
QWEN_QUANTIZATION = "bnb-nf4"
FALLBACK_CODE = KAGGLE_INPUT / "dependency-aware-selective-regeneration-code"
FALLBACK_DATA = KAGGLE_INPUT / "dependency-aware-selective-regeneration-data"

def discover(label, candidates, required_subdir=None):
    for p in candidates:
        if p.is_dir():
            if required_subdir is None or (p / required_subdir).is_dir():
                return p
            print(f"  [info] {p.name} exists but missing '{required_subdir}'")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if entry.is_dir() and (required_subdir is None or (entry / required_subdir).is_dir()):
                return entry
    raise FileNotFoundError(f"Cannot find {label} in {KAGGLE_INPUT}")

CODE_DIR = discover("code dataset", [KNOWN_CODE, FALLBACK_CODE], required_subdir="src")

DATA_DIR = discover("data dataset", [KNOWN_DATA, FALLBACK_DATA], required_subdir="scenarios")

src_dir = CODE_DIR / "src"
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir))
else:
    raise FileNotFoundError(f"src/ not found in code dataset: {CODE_DIR}")


MODEL_WEIGHT_SUFFIXES = (".safetensors", ".bin")

def _has_weight_files(p: Path) -> bool:
    return any(
        f.is_file() and f.suffix in MODEL_WEIGHT_SUFFIXES
        for f in p.rglob("*")
    )

def _is_valid_model_dir(p: Path) -> bool:
    return p.is_dir() and (p / "config.json").is_file() and _has_weight_files(p)

def discover_model(candidates) -> Path:
    for p in candidates:
        if _is_valid_model_dir(p):
            return p
        print(f"  [info] {p.name}: not a valid Qwen model dir (config.json + weights required)")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if _is_valid_model_dir(entry):
                return entry
    raise FileNotFoundError(
        f"Cannot find a valid Qwen model under {KAGGLE_INPUT}: "
        "config.json and at least one .safetensors/.bin weight file required"
    )

MODEL_CANDIDATES = [KNOWN_MODEL] if KNOWN_MODEL.is_dir() else []
MODEL_PATH = str(discover_model(MODEL_CANDIDATES).resolve())

SCRIPT_PATH = CODE_DIR / "seven_arm_benchmark.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"seven_arm_benchmark.py not found in {CODE_DIR}")


SOURCE_COMMIT = "7f2a4509482dc7e62c2b243374592e9a88e2ff48"

DEPLOYED_BUILD_ID = "7f2a450"

HF_RESULTS_REPO_ID = "NabilDo/selective-regeneration-experiment-results"

# ---- FULL9-EXEC-01: single in-notebook source of truth for operational paths
# All current operational paths derive from KAGGLE_DEPLOYMENT_PATHS. No generic
# OUTPUT_DIR or canary/continuous execution state is kept.
KAGGLE_DEPLOYMENT_PATHS = {
    "working_root": Path("/kaggle/working"),
    "runs_root": Path("/kaggle/working/runs"),
    "full9_run_dir_name": "qwen14b_bnb_nf4_full9_scientific_smoke_wsfix_7f2a450",
    "full9_output_dir": Path(
        "/kaggle/working/runs/qwen14b_bnb_nf4_full9_scientific_smoke_wsfix_7f2a450"
    ),
    "preflight_output_dir": Path(
        "/kaggle/working/runs/preflight_full9_wsfix_7f2a450"
    ),
    "runtime_env_dir": Path("/kaggle/working/runs/environment"),
    "full9_console_name": "kaggle_console.log",
    "preflight_console_name": "kaggle_preflight_console.log",
    "evidence_archive_root": Path("/kaggle/working"),
    "evidence_archive_stem": "corrected-full9-wsfix-7f2a450",
}
FULL9_OUTPUT_DIR = KAGGLE_DEPLOYMENT_PATHS["full9_output_dir"]

print(f"Full-9 output dir: {FULL9_OUTPUT_DIR}")
print(f"Source commit:     {SOURCE_COMMIT}")
print(f"Build ID:          {DEPLOYED_BUILD_ID}")
print(f"Model path:        {MODEL_PATH}")

# ---- R7B: live-run observability -------------------------------------------

class ScientificSmokeExecutionError(RuntimeError):
    'Raised when a benchmark invocation does not produce a valid result.'

SCIENTIFIC_FAILURE_KINDS = frozenset({
    "model_output",
    "build",
    "changed_requirement",
    "regression",
    "architecture",
    "scientific_budget_exhausted",
})

ENGINEERING_FAILURE_KINDS = frozenset({
    "infrastructure",
    "infrastructure_nonrepairable",
    "harness_defect",
    "timeout",
    "environment",
    "environment_preflight",
})

EVIDENCE_FILES = (
    "experiment_id.txt",
    "source_identity.json",
    "environment_metadata.json",
    "checkpoint.json",
    "progress.json",
    "failure_records.json",
    "remote_sync.json",
    "run_records.jsonl",
    "dashboard/dashboard_summary.json",
    "dashboard/run_matrix.csv",
    "dashboard/strategy_summary.csv",
    "dashboard/failure_summary.csv",
)

def _load_smoke_evidence(output_dir):
    output_dir = Path(output_dir)
    evidence = {}
    for name in EVIDENCE_FILES:
        p = output_dir / name
        if not p.is_file():
            evidence[name] = None
            continue
        try:
            if name.endswith(".jsonl"):
                rows = [_json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
                evidence[name] = rows
            elif name.endswith(".json"):
                evidence[name] = _json.loads(p.read_text(encoding="utf-8"))
            else:
                evidence[name] = p.read_text(encoding="utf-8").strip()
        except Exception as exc:
            evidence[name] = {"load_error": str(exc)}
    return evidence

def _terminal_record_outcome(record):
    status = str(record.get("status", ""))
    if status == "succeeded":
        return "scientific_success"
    if status in ("timed_out", "cancelled") or status != "failed":
        return "engineering_blocker"
    kinds = {
        str(item.get("kind", ""))
        for item in (record.get("failure_details") or [])
        if isinstance(item, dict) and item.get("kind")
    }
    classification = str(record.get("failure_classification", ""))
    if classification:
        kinds.add(classification)
    if not kinds or kinds & ENGINEERING_FAILURE_KINDS:
        return "engineering_blocker"
    if kinds <= SCIENTIFIC_FAILURE_KINDS:
        return "scientific_failure"
    return "engineering_blocker"

def _extract_evidence_root_cause(record):
    import re
    candidates = []
    for detail in (record.get("failure_details") or []):
        if not isinstance(detail, dict):
            continue
        for key in ("details", "message", "error"):
            value = detail.get(key)
            if value:
                candidates.append(str(value))
    combined = "\n".join(candidates)
    matches = re.findall(
        r"^([A-Za-z_][A-Za-z0-9_.]*(?:Error|Exception)):\s*(.*)$",
        combined,
        flags=re.MULTILINE,
    )
    if matches:
        kind, message = matches[-1]
        return f"{kind}: {message}".strip()
    lines = [line.strip() for line in combined.splitlines() if line.strip()]
    return lines[-1] if lines else "(root cause unavailable)"

def _gpu_diagnostics():
    try:
        import torch
        if not torch.cuda.is_available():
            return "cuda unavailable"
        name = torch.cuda.get_device_name(0)
        alloc = torch.cuda.memory_allocated(0) / 2**30
        reserved = torch.cuda.memory_reserved(0) / 2**30
        return f"{name} allocated={alloc:.2f}GiB reserved={reserved:.2f}GiB"
    except Exception as exc:
        return f"gpu diagnostics unavailable: {exc}"

def _label_bar_containers(ax):
    for container in getattr(ax, "containers", ()) or ():
        values = [float(v) for v in (getattr(container, "datavalues", None) or ())]
        labels = [f"{v:,.0f}" for v in values]
        ax.bar_label(container, labels=labels, padding=3)

def _display_smoke_dashboard(output_dir):
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    records = evidence.get("run_records.jsonl") or []
    if not isinstance(records, list):
        records = []
    cp = evidence.get("checkpoint.json") or {}
    if not isinstance(cp, dict):
        cp = {}
    dash_dir = output_dir / "dashboard"
    dash_dir.mkdir(parents=True, exist_ok=True)
    print("\n--- SMOKE DASHBOARD ---")
    print("KPI: planned=%s completed=%s status=%s" % (
        cp.get("total_planned"), cp.get("total_completed"), cp.get("completion_status")))
    model_calls = sum(int(r.get("total_workflow_model_calls", 0) or 0) for r in records)
    tokens = sum(int(r.get("total_workflow_tokens", 0) or 0) for r in records)
    print("KPI: model_calls=%s tokens=%s records=%s" % (model_calls, tokens, len(records)))
    try:
        import pandas as pd
        rows = []
        for r in records:
            rows.append({
                "run_id": r.get("run_id", ""),
                "scenario": r.get("scenario_id", ""),
                "strategy": r.get("strategy_id", r.get("strategy_name", "")),
                "status": r.get("status", ""),
                "model_calls": int(r.get("total_workflow_model_calls", 0) or 0),
                "tokens": int(r.get("total_workflow_tokens", 0) or 0),
                "duration_seconds": float(r.get("duration_seconds", 0) or 0),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            print("\nPer-run table:")
            print(df.to_string(index=False))
            print("\nScenario x Strategy matrix (status):")
            try:
                matrix = df.pivot_table(index="scenario", columns="strategy", values="status", aggfunc="first")
                print(matrix.to_string())
            except Exception as exc:
                print("(matrix unavailable: %s)" % exc)
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        if not df.empty:
            try:
                ax = df.groupby(["strategy", "status"]).size().unstack(fill_value=0).plot(kind="bar", title="status")
                _label_bar_containers(ax)
                fig = ax.get_figure()
                fig.savefig(dash_dir / "status_by_strategy.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "status_by_strategy.png"))
            except Exception as exc:
                print("(chart status_by_strategy.png failed: %s)" % exc)
            try:
                ax = df.groupby("strategy")["tokens"].sum().plot(kind="bar", title="tokens")
                _label_bar_containers(ax)
                fig = ax.get_figure()
                fig.savefig(dash_dir / "tokens_by_strategy.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "tokens_by_strategy.png"))
            except Exception as exc:
                print("(chart tokens_by_strategy.png failed: %s)" % exc)
            try:
                ax = df.groupby("strategy")["model_calls"].sum().plot(kind="bar", title="model_calls")
                _label_bar_containers(ax)
                fig = ax.get_figure()
                fig.savefig(dash_dir / "model_calls_by_strategy.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "model_calls_by_strategy.png"))
            except Exception as exc:
                print("(chart model_calls_by_strategy.png failed: %s)" % exc)
            try:
                ax = df.groupby("strategy")["duration_seconds"].sum().plot(kind="bar", title="duration_seconds")
                _label_bar_containers(ax)
                fig = ax.get_figure()
                fig.savefig(dash_dir / "duration_by_strategy.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "duration_by_strategy.png"))
            except Exception as exc:
                print("(chart duration_by_strategy.png failed: %s)" % exc)
        failed = [r for r in records if r.get("status") != "succeeded"]
        if failed:
            causes = {}
            for r in failed:
                k = r.get("failure_classification") or r.get("failure_stage") or "unknown"
                causes[k] = causes.get(k, 0) + 1
            print("\nFailure causes: %s" % causes)
            try:
                fig, ax = plt.subplots()
                ax.bar(list(causes.keys()), list(causes.values()))
                ax.set_title("failure_causes")
                _label_bar_containers(ax)
                fig.savefig(dash_dir / "failure_causes.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "failure_causes.png"))
            except Exception as exc:
                print("(failure_causes.png failed: %s)" % exc)
    except Exception as exc:
        print("(dashboard tables/charts unavailable: %s)" % exc)

def _raise_actionable_smoke_error(output_dir):
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    print("\n" + "=" * 78)
    print("BENCHMARK EXECUTION BLOCKED - actionable diagnosis")
    print("=" * 78)
    _display_smoke_dashboard(output_dir)
    cp = evidence.get("checkpoint.json") or {}
    if not isinstance(cp, dict):
        cp = {}
    records = evidence.get("run_records.jsonl") or []
    if not isinstance(records, list):
        records = []
    sync = evidence.get("remote_sync.json") or {}
    if not isinstance(sync, dict):
        sync = {}
    first = records[-1] if records else {}
    details = [d for d in (first.get("failure_details") or []) if isinstance(d, dict)]
    primary = next(
        (d for d in reversed(details) if d.get("stage") not in (None, "regeneration")),
        details[-1] if details else {},
    )
    experiment_id = evidence.get("experiment_id.txt") or cp.get("experiment_id") or "?"
    lines = []
    lines.append("experiment/source/build: %s / %s / %s" % (
        experiment_id,
        cp.get("source_commit", "?"),
        cp.get("deployed_build_id", "?"),
    ))
    lines.append("latest terminal run: %s" % first.get("run_id", "?"))
    lines.append("scenario: %s" % first.get("scenario_id", "?"))
    lines.append("strategy: %s" % (first.get("strategy_id") or first.get("strategy_name") or "?"))
    lines.append("status/outcome: %s / %s" % (
        first.get("status", "?"), _terminal_record_outcome(first)))
    lines.append("stage: %s" % primary.get("stage", "?"))
    lines.append("classification: %s" % first.get("failure_classification", "?"))
    lines.append("root cause: %s" % _extract_evidence_root_cause(first))
    msgs = []
    for r in records:
        for d in (r.get("failure_details") or []):
            if not isinstance(d, dict):
                continue
            m = d.get("message") or d.get("error") or ""
            if m and m not in msgs:
                msgs.append(m)
    if msgs:
        lines.append("top unique messages:")
        for item in msgs[:5]:
            compact = " ".join(str(item).split())
            lines.append("  - " + compact[:800])
    calls = sum(int(r.get("total_workflow_model_calls", 0) or 0) for r in records)
    tokens = sum(int(r.get("total_workflow_tokens", 0) or 0) for r in records)
    lines.append("model calls: %s" % calls)
    lines.append("tokens: %s" % tokens)
    sel = sum(int(r.get("selected_artifact_count", 0) or 0) for r in records)
    regen = sum(int(r.get("regenerated_artifact_count", 0) or 0) for r in records)
    lines.append("selected/regenerated: %s/%s" % (sel, regen))
    lines.append("GPU/OOM: %s" % _gpu_diagnostics())
    lines.append("HF state: %s" % sync.get("last_sync", "?"))
    lines.append("evidence paths:")
    for name in EVIDENCE_FILES:
        lines.append("  " + str(output_dir / name))
    lines.append(
        "next action: fix only the engineering blocker above. A scientific "
        "model/code failure is a terminal benchmark result and must not be "
        "treated as a notebook execution error."
    )
    msg = "\n".join(lines)
    print("\n--- ACTIONABLE ENGINEERING ERROR ---")
    print(msg)
    raise ScientificSmokeExecutionError(msg)

def _run_benchmark_live(exec_cmd, output_dir, tail_limit=200):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    console_path = output_dir / KAGGLE_DEPLOYMENT_PATHS["full9_console_name"]

    sub_env = os.environ.copy()
    if not sub_env.get("HF_TOKEN", "").strip():
        raise RuntimeError("HF_TOKEN was not propagated to subprocess environment")
    sub_env["PYTHONPATH"] = str(CODE_DIR / "src") + (
        os.pathsep + sub_env["PYTHONPATH"] if sub_env.get("PYTHONPATH") else ""
    )
    sub_env["PYTHONUNBUFFERED"] = "1"
    sub_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    sub_env["TOKENIZERS_PARALLELISM"] = "false"

    print("Running:", " ".join(str(x) for x in exec_cmd))
    print("\n--- Output ---")
    tail = []
    with console_path.open("a", encoding="utf-8") as console:
        proc = subprocess.Popen(
            exec_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=sub_env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            console.write(line)
            tail.append(line)
            if len(tail) > tail_limit:
                tail.pop(0)
    return_code = proc.wait()
    print(f"\nReturn code: {return_code}")
    if return_code != 0:
        _raise_actionable_smoke_error(output_dir)
    return tail

# ---- FULL9-EXEC-01: corrected Full-9 executable notebook contract -----------
# A single execution cell runs the exact 3x3 plan (3 scenarios x 3 strategies);
# a fail-closed verifier then requires the exact matrix with pinned identity.
EXPECTED_MODEL_IDENTITY = "qwen:14b-instruct-v1:bnb-nf4:cfg-cc9474140d25"
EXPECTED_PROFILE = "scientific-smoke-v2"
EXPECTED_PROTOCOL_VERSION = "1.0"
FULL9_EXPECTED_MATRIX = frozenset(
    (scenario_id, strategy_id)
    for scenario_id in ("todo-smoke-001", "todo-smoke-002", "todo-smoke-003")
    for strategy_id in ("monolithic", "selective", "iterative_repository_agent")
)

def _verify_full9_evidence(output_dir):
    """FULL9-EXEC-01 fail-closed guardrail: exact 3x3 matrix + pinned identity.

    Terminal semantics (F4): a record is ACCEPTED when it is ``succeeded``, or
    when it is ``failed`` with terminal outcome ``scientific_failure``. Any
    engineering blocker, infrastructure/harness/environment failure, timeout,
    cancellation, or malformed/non-terminal outcome is REJECTED.
    """
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    cp = evidence.get("checkpoint.json")
    if not isinstance(cp, dict):
        raise ScientificSmokeExecutionError(
            "Full-9 verification FAILED: checkpoint.json missing or invalid"
        )
    records = evidence.get("run_records.jsonl")
    if not isinstance(records, list):
        raise ScientificSmokeExecutionError(
            "Full-9 verification FAILED: run_records.jsonl missing or invalid"
        )
    identity = evidence.get("source_identity.json")
    if not isinstance(identity, dict):
        identity = {}
    matrix = {(r.get("scenario_id"), r.get("strategy_id")) for r in records}
    matrix_ok = len(records) == 9 and matrix == FULL9_EXPECTED_MATRIX
    terminal_ok = all(
        r.get("status") == "succeeded"
        or (
            r.get("status") == "failed"
            and _terminal_record_outcome(r) == "scientific_failure"
        )
        for r in records
    )
    pending_ids = cp.get("pending_run_ids") or []
    checks = {
        "checkpoint source identity": cp.get("source_commit") == SOURCE_COMMIT,
        "checkpoint build identity": cp.get("deployed_build_id") == DEPLOYED_BUILD_ID,
        "checkpoint model identity": cp.get("model_identity") == EXPECTED_MODEL_IDENTITY,
        "checkpoint profile": cp.get("profile") == EXPECTED_PROFILE,
        "checkpoint protocol": cp.get("protocol_version") == EXPECTED_PROTOCOL_VERSION,
        "completion status is completed": cp.get("completion_status") == "completed",
        "total planned is 9": cp.get("total_planned") == 9,
        "total completed is 9": cp.get("total_completed") == 9,
        "checkpoint pending is 0": len(pending_ids) == 0,
        "exact 3x3 scenario x strategy matrix": matrix_ok,
        "all records terminal scientific outcomes": terminal_ok,
        "record source identity": all(
            r.get("source_commit") == SOURCE_COMMIT for r in records
        ),
        "record profile": all(r.get("profile") == EXPECTED_PROFILE for r in records),
        "experiment id present": bool((evidence.get("experiment_id.txt") or "").strip()),
        "source identity model": identity.get("model_identity") == EXPECTED_MODEL_IDENTITY,
        "source identity profile": identity.get("profile") == EXPECTED_PROFILE,
        "source identity protocol": identity.get("protocol_version") == EXPECTED_PROTOCOL_VERSION,
        "source identity commit": identity.get("source_commit") == SOURCE_COMMIT,
        "source identity build": identity.get("deployed_build_id") == DEPLOYED_BUILD_ID,
    }
    problems = [label for label, ok in checks.items() if not ok]
    if problems:
        outcomes = {_terminal_record_outcome(r) for r in records}
        raise ScientificSmokeExecutionError(
            "Full-9 verification FAILED: " + "; ".join(problems)
            + " | engineering blocker record: %s" % ("engineering_blocker" in outcomes)
            + " | scientific_failed: %s" % ("scientific_failure" in outcomes)
            + " | exact 3x3 scenario x strategy matrix: %s" % matrix_ok
        )
    print("FULL9 VERIFICATION: PASSED - exact 3x3 scenario x strategy matrix")
    return matrix

def _export_full9_evidence(output_dir, archive_root=None, bundle_stem=None, timestamp=None):
    """FULL9-EXEC-01: archive the ENTIRE corrected Full-9 output tree.

    The zip carries exactly one top-level Full-9 directory and must not
    include any sibling canary/preflight/runs content.
    """
    output_dir = Path(output_dir)
    if not output_dir.is_dir():
        raise FileNotFoundError(f"Full-9 output dir missing: {output_dir}")
    archive_root = Path(archive_root) if archive_root else Path(
        KAGGLE_DEPLOYMENT_PATHS["evidence_archive_root"]
    )
    bundle_stem = bundle_stem or KAGGLE_DEPLOYMENT_PATHS["evidence_archive_stem"]
    stamp = timestamp or datetime.now().strftime("%Y-%m-%d-%H%M%S")
    archive_root.mkdir(parents=True, exist_ok=True)
    bundle_path = archive_root / f"{bundle_stem}-{stamp}.zip"
    top_dir = output_dir.name
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, f"{top_dir}/{p.relative_to(output_dir).as_posix()}")
    evidence = _load_smoke_evidence(output_dir)
    sync = evidence.get("remote_sync.json") or {}
    if not isinstance(sync, dict):
        sync = {}
    print("FULL9 EVIDENCE EXPORT: PASSED - %s" % bundle_path)
    print("HF sync state:", {
        "last_sync": sync.get("last_sync"),
        "timestamp": sync.get("timestamp"),
        "remote_path": sync.get("remote_path"),
        "details": sync.get("details"),
    })
    return bundle_path


In [ ]:
# R7C correction: install the exact pinned runtime lock and verify imports/versions.
import importlib
import importlib.metadata

LOCK_PATH = CODE_DIR / "requirements-smoke-kaggle.lock"
if not LOCK_PATH.is_file():
    raise FileNotFoundError(f"pinned runtime lock missing in code dataset: {LOCK_PATH}")

PYTHON_RUNTIME = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError(
        f"Unsupported Python runtime {PYTHON_RUNTIME}; expected Python 3.11 or 3.12"
    )
print(f"Python runtime: {PYTHON_RUNTIME} [OK]")

print("Installing exact pinned Smoke runtime lock ...")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(LOCK_PATH)],
    text=True,
)
if install.returncode != 0:
    raise RuntimeError("pip install of requirements-smoke-kaggle.lock failed")

EXPECTED_RUNTIME = {
    "django": ("Django", "django", "5.2.16"),
    "djangorestframework": ("djangorestframework", "rest_framework", "3.17.1"),
    "pytest": ("pytest", "pytest", "8.4.2"),
    "pytest_django": ("pytest-django", "pytest_django", "4.12.0"),
    "accelerate": ("accelerate", "accelerate", "1.14.0"),
    "bitsandbytes": ("bitsandbytes", "bitsandbytes", "0.49.2"),
    "transformers": ("transformers", "transformers", "4.57.6"),
}

print("\n--- PINNED RUNTIME VERSION TABLE ---")
mismatches = []
runtime_versions = {}
for key, (distribution, module_name, expected) in EXPECTED_RUNTIME.items():
    try:
        importlib.import_module(module_name)
        actual = importlib.metadata.version(distribution)
    except Exception as exc:
        actual = f"NOT_INSTALLED ({exc.__class__.__name__})"
    runtime_versions[key] = actual
    ok = "OK" if actual == expected else "MISMATCH"
    print(f"  {key:20s} expected={expected} actual={actual} [{ok}]")
    if actual != expected:
        mismatches.append(f"{key}={actual} (expected {expected})")
if mismatches:
    raise RuntimeError("pinned runtime version mismatch: " + "; ".join(mismatches))

env_meta = {
    "schema": "kaggle_runtime_environment.v1",
    "source_commit": SOURCE_COMMIT,
    "python_version": PYTHON_RUNTIME,
    "runtime_versions": runtime_versions,
}
RUNTIME_META_DIR = KAGGLE_DEPLOYMENT_PATHS["runtime_env_dir"]
RUNTIME_META_DIR.mkdir(parents=True, exist_ok=True)
(RUNTIME_META_DIR / "runtime_environment.json").write_text(
    _json.dumps(env_meta, indent=2, sort_keys=True), encoding="utf-8")
print("\nRUNTIME INSTALL + VERIFICATION: PASSED")


## R7C-REAL-RUN-ROOT-CLOSURE — Kaggle smoke preflight gate

This cell runs `--kaggle-preflight-only` before any experiment is created. It validates:

1. **Pinned runtime** — exact installed versions of Django, DRF, pytest, pytest-django, accelerate, bitsandbytes, torch, transformers
2. **Baseline Todo workspace** — `manage.py check` + `makemigrations todo --check --dry-run`
3. **Qwen 14B BNB-NF4 load** — `Qwen2.5-Coder-14B-Instruct` base checkpoint via BitsAndBytes NF4: `load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_compute_dtype=float16`, `bnb_4bit_use_double_quant=True`, `device_map="auto"`, Transformers 4.57.6
4. **Deterministic 64-token probe** — seeded `torch.manual_seed(0)`
5. **VRAM headroom** — >= 2.0 GiB free after the probe

On failure it raises before any experiment, RunRecord, workspace result, or HF state is created. The machine-readable result is written to `kaggle_smoke_preflight.v1.json`.


In [ ]:
# R7C independent audit: stream the engineering preflight live and persist its log.
PREFLIGHT_DIR = KAGGLE_DEPLOYMENT_PATHS["preflight_output_dir"]
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_CONSOLE = PREFLIGHT_DIR / KAGGLE_DEPLOYMENT_PATHS["preflight_console_name"]

preflight_cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--kaggle-preflight-only",
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--qwen-quantization", QWEN_QUANTIZATION,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(PREFLIGHT_DIR),
]
preflight_env = os.environ.copy()
preflight_env["PYTHONPATH"] = str(CODE_DIR / "src") + (
    os.pathsep + preflight_env["PYTHONPATH"]
    if preflight_env.get("PYTHONPATH")
    else ""
)
preflight_env["PYTHONUNBUFFERED"] = "1"
preflight_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
preflight_env["TOKENIZERS_PARALLELISM"] = "false"

print("Running preflight:", " ".join(str(x) for x in preflight_cmd))
print("\n--- PREFLIGHT OUTPUT ---")
with PREFLIGHT_CONSOLE.open("a", encoding="utf-8") as console:
    preflight = subprocess.Popen(
        preflight_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=preflight_env,
    )
    assert preflight.stdout is not None
    for line in preflight.stdout:
        print(line, end="", flush=True)
        console.write(line)
preflight_return_code = preflight.wait()
print(f"\nPreflight return code: {preflight_return_code}")
if preflight_return_code != 0:
    raise RuntimeError(
        "KAGGLE SMOKE PREFLIGHT FAILED - no experiment will be started. "
        f"Inspect {PREFLIGHT_CONSOLE} and kaggle_smoke_preflight.v1.json."
    )
print("KAGGLE SMOKE PREFLIGHT: PASSED")


## FULL9-EXEC-01 - single Full-9 execution cell

The `full9-execution-cell` runs the entire 9-run plan (3 scenarios x 3
strategies) in a single invocation. There is no `--max-runs 1` resume loop and
no `--auto-resume-hf`; the benchmark owns the exact 3x3 matrix end to end.
Before launching, it refuses to reuse a non-empty Full-9 output dir.

After the run, `full9-verification-cell` applies the fail-closed
`_verify_full9_evidence` guardrail: it requires the exact 3x3 scenario x
strategy matrix, checkpoint planned=9/completed=9/pending=0, every identity
field pinned to `SOURCE_COMMIT` / `DEPLOYED_BUILD_ID` /
`EXPECTED_MODEL_IDENTITY`, and every record terminal — `succeeded` or a
`failed` scientific failure. Engineering blockers, infrastructure/harness/
environment failures, timeouts, cancellations, and malformed outcomes are
rejected.

`export-evidence-cell` archives the ENTIRE corrected Full-9 output tree to a
single-top-level zip under `/kaggle/working/`.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token or not hf_token.strip():
    raise RuntimeError("HF_TOKEN Kaggle secret is missing or blank")

os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN: retrieved and set in environment")

In [ ]:
# FULL9-EXEC-01: run the exact 3x3 plan (3 scenarios x 3 strategies) in one shot.
# No --max-runs, no --strategy, no --auto-resume-hf: the Full-9 cell is the
# single authority that runs all 9 planned arms end to end.
#
# Fail-closed output guard (F1): an already-populated Full-9 output dir must
# never be reused silently. The dir is refused before any subprocess launch;
# a non-empty dir is never cleaned or deleted automatically.
if FULL9_OUTPUT_DIR.exists() and any(FULL9_OUTPUT_DIR.iterdir()):
    raise RuntimeError(
        "FULL9 FAIL-CLOSED OUTPUT GUARD: refusing to reuse a non-empty "
        "Full-9 output dir: %s" % FULL9_OUTPUT_DIR
    )
exec_cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--qwen-quantization", QWEN_QUANTIZATION,
    "--max-attempts", "3",
    "--protocol-version", "1.0",
    "--max-completion-tokens-per-call", "1024",
    "--max-total-workflow-tokens", "0",
    "--timeout", "300",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-commit", SOURCE_COMMIT,
    "--deployed-build-id", DEPLOYED_BUILD_ID,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(FULL9_OUTPUT_DIR),
    "--hf-sync",
    "--new-experiment",
]
_run_benchmark_live(exec_cmd, FULL9_OUTPUT_DIR)
print("FULL9 EXECUTION: 9/9 runs started")


In [ ]:
# FULL9-EXEC-01: fail-closed post-run guardrail over the exact 3x3 matrix.
_verify_full9_evidence(FULL9_OUTPUT_DIR)
print("FULL9 VERIFICATION CELL: PASSED")


In [ ]:
# FULL9-EXEC-01: export the ENTIRE corrected Full-9 output tree (F5 contract).
# The archive contains exactly one top-level Full-9 directory; sibling canary
# and preflight runs are never included.
_export_full9_evidence(FULL9_OUTPUT_DIR)


## Notes

- **Scientific Smoke V2 (FULL9-EXEC-01)**: 1 repo (todo) x 3 scenarios x 3 arms x 1 run = 9 runs, non-publication.
- All outputs go to `/kaggle/working/runs/qwen14b_bnb_nf4_full9_scientific_smoke_wsfix_7f2a450/`.
- Operational paths derive from `KAGGLE_DEPLOYMENT_PATHS` in setup-cell.
- Evidence export archives the ENTIRE Full-9 output tree as `corrected-full9-wsfix-7f2a450-<timestamp>.zip`.
- Internet is required for Hugging Face result synchronization.
- `HF_TOKEN` is required and read from Kaggle Secrets.
- Qwen model loading remains offline from the attached Kaggle Model.
- The full9-execution-cell runs the full 3x3 matrix with `--new-experiment`.
